# Part 0 — Stigma Classification from `clean_text`
Generate fear / humor / urgency / neutral stigma labels using zero-shot classification.

## Stigma Classification Method

Stigma labels were assigned to individual posts using a **rule-based, lexicon-driven hierarchical classifier**, designed to maximize precision, interpretability, and temporal stability. This approach was adopted after empirical evaluation showed that zero-shot large language models produced inconsistent and semantically unstable stigma assignments at scale.

### Overview

Each post was classified into one of four mutually exclusive stigma categories:

- **Fear**
- **Urgency**
- **Humor**
- **Neutral**

Classification proceeds in a **fixed hierarchical order**, ensuring that high-precision categories are resolved first and that ambiguous cases default conservatively.

### Text Preprocessing

All posts were preprocessed as follows:

1. Missing text values were filled with empty strings.
2. Text was lowercased.
3. Tokens were extracted using a lightweight regular expression (`[a-zA-Z']+`), producing a set of unique word tokens per post.

This minimalist tokenization was intentionally chosen to preserve scalability and avoid introducing dependencies on external NLP pipelines.

### Lexicon Construction

Four curated keyword lexicons were defined to capture domain-specific stigma signals:

- **Fear lexicon**: terms related to danger, mortality, disease severity, and risk amplification (e.g., *panic, deadly, outbreak, fatal*).
- **Blame lexicon**: attribution of responsibility or fault (e.g., *government failure, lab leak, negligence*).
- **Urgency lexicon**: language indicating immediacy or calls to action (e.g., *urgent, breaking, act now*).
- **Humor lexicon**: explicit markers of humor or sarcasm (e.g., *lol, meme, sarcastic*).

Blame was conceptually treated as a subtype of fear, consistent with stigma theory, and was merged into the **fear** category.

### Hierarchical Classification Rules

Each post was classified using the following ordered rules:

1. **Humor detection (highest precision)**  
   If any humor-related keyword was present, the post was labeled *humor*.  
   This rule was applied first to prevent sarcasm from being misclassified as fear or urgency.

2. **Blame attribution → Fear merge**  
   Posts containing blame-related terms were labeled *fear*, reflecting threat attribution dynamics.

3. **Urgency detection**  
   Posts containing urgency-related terms were labeled *urgency*.

4. **Fear keyword detection**  
   Posts containing fear-related terms were labeled *fear*.

5. **Sentiment-based fallback (VADER)**  
   For posts not captured by keyword rules, sentiment polarity was computed using the VADER sentiment analyzer.  
   Posts were labeled *fear* if:
   - Negative sentiment exceeded 0.4, or
   - Compound sentiment score was below −0.3.

6. **Neutral default**  
   Posts not meeting any prior criteria were labeled *neutral*.

This hierarchy ensures deterministic, reproducible labeling and avoids label oscillation across similar texts.

### Sentiment Analysis Integration

Sentiment polarity scores were computed using the **VADER sentiment analyzer**, which is optimized for short, informal text. Sentiment was not used as a primary classifier but rather as a **fallback mechanism** to capture implicit fear signals not explicitly expressed through keywords.

### Encoding and Output

Final stigma labels were one-hot encoded into the following binary features:

- `stigma_fear`
- `stigma_humor`
- `stigma_urgency`
- `stigma_neutral`

All missing categories were explicitly filled with zeros to ensure schema consistency. The enriched dataset was then exported for downstream temporal aggregation and lead–lag feature engineering.

### Rationale for Rule-Based Approach

This rule-based classifier was favored over zero-shot approaches because it:

- Produces **stable labels across time**
- Is **fully interpretable and auditable**
- Avoids semantic drift from pretrained LLM priors
- Scales efficiently to millions of posts
- Aligns closely with soc


In [6]:
import pandas as pd
import re
import nltk

# download resources once
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sid = SentimentIntensityAnalyzer()

# ---- 1. Keyword lexicons ----
fear_words = set("""
fear scared terrified panic panic-buying worried danger outbreak deadly virus death fatal severe emergency risk warning 
alarming threat spread transmission pathogen contagious
""".split())

blame_words = set("""
government failure corrupt incompetent irresponsible blame fault caused by lab leak negligence mistake 
""".split())

urgency_words = set("""
urgent breaking alert immediate act now warning happening right now update emergency attention
""".split())

humor_words = set("""
lol lmao rofl haha funny joke sarcastic meme hilarious clown
""".split())

def classify_stigma(text):
    t = text.lower()

    # Tokenize quickly
    words = set(re.findall(r"[a-zA-Z']+", t))

    # ---- RULE 1: humor detection (highest precision) ----
    if len(words & humor_words) > 0:
        return "humor"

    # ---- RULE 2: blame → fear merge ----
    if len(words & blame_words) > 0:
        return "fear"

    # ---- RULE 3: urgency ----
    if len(words & urgency_words) > 0:
        return "urgency"

    # ---- RULE 4: fear words ----
    if len(words & fear_words) > 0:
        return "fear"

    # ---- RULE 5: sentiment-based fallback ----
    ss = sid.polarity_scores(t)
    if ss['neg'] > 0.4 or ss['compound'] < -0.3:
        return "fear"

    # ---- default ----
    return "neutral"


# Apply to your dataset
df = pd.read_csv("../merged_output_with_inferred_country.csv")
df["clean_text"] = df["clean_text"].fillna("").astype(str)

df["stigma_label"] = df["clean_text"].apply(classify_stigma)

# One-hot encode
dummies = pd.get_dummies(df["stigma_label"], prefix="stigma")
for col in ["stigma_fear", "stigma_humor", "stigma_urgency", "stigma_neutral"]:
    if col not in dummies:
        dummies[col] = 0

df = pd.concat([df, dummies], axis=1)

df.to_csv("merged_output_with_stigma.csv", index=False)

df.head()


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/gazimahmud/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


,activities,content_type,creation_time,id,is_branded_content,lang,link_attachment.caption,link_attachment.description,link_attachment.link,link_attachment.name,...,surface.username,text,clean_text,inferred_country,inferred_country_confidence,stigma_label,stigma_fear,stigma_humor,stigma_neutral,stigma_urgency
0,NaN,status,2025-11-10T19:00:19+00:00,2980185815499999,False,en,NaN,NaN,NaN,NaN,...,amtribnews,"Despite Owners Exhausting Legal Appeals, The C...","despite owners exhausting legal appeals, the c...",United States,0.50,fear,True,False,False,False
1,NaN,albums,2025-11-10T18:23:30+00:00,686591307858179,False,ro,NaN,NaN,NaN,NaN,...,IShouldBeSoLacheLacheLachee,Convorbiri literare 1. Fata noastra lasconista...,convorbiri literare 1. fata noastra lasconista...,Romania,0.40,fear,True,False,False,False
2,NaN,videos,2025-11-10T18:14:12+00:00,1006177595025072,False,en,NaN,NaN,NaN,NaN,...,maria.gerke,"Is this the face of evil? Apparently, this man...","is this the face of evil? apparently, this man...",Italy,0.05,fear,True,False,False,False
3,NaN,links,2025-11-10T16:24:10+00:00,840171035327111,False,en,brecon-radnor.co.uk,NaN,https://www.brecon-radnor.co.uk/news/farming/a...,Avian influenza confirmed at Powys premises,...,BreconRadnorExpress,A case of Highly Pathogenic Avian Influenza (H...,a case of highly pathogenic avian influenza (h...,Spain,0.30,neutral,False,False,True,False
4,NaN,links,2025-11-10T16:05:06+00:00,1028189766103347,False,de,promisundmehr.de,"Promis, Prominente, Stars und Sternchen ... Di...",https://www.promisundmehr.de/vogelgrippe-bei-h...,Vogelgrippe bei Haustieren: Katzen sind potenz...,...,promisundmehr,Diese Symptome zeigen Katzen bei Vogelgrippe\n...,diese symptome zeigen katzen bei vogelgrippe\n...,Germany,0.50,fear,True,False,False,False


### Zero shot alternative - do not execute

In [ ]:

# Install dependencies if needed
#!pip install transformers torch pandas tqdm

import pandas as pd
from transformers import pipeline
from tqdm import tqdm

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Load dataset
df = pd.read_csv("../merged_output_with_inferred_country.csv")
df['clean_text'] = df['clean_text'].fillna("").astype(str)

# Zero-shot classifier
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

labels = ["fear", "humor", "urgency", "neutral", "blame"]

stigma_outputs = []
for text in tqdm(df['clean_text'], desc="Classifying stigma"):
    result = classifier(text, candidate_labels=labels)
    stigma_outputs.append(result['labels'][0])  # top predicted label

df['stigma_raw'] = stigma_outputs

# Merge blame -> fear
df['stigma_label'] = df['stigma_raw'].replace({"blame":"fear"})

# One-hot encoding
dummies = pd.get_dummies(df['stigma_label'], prefix="stigma")
for col in ["stigma_fear","stigma_humor","stigma_urgency","stigma_neutral"]:
    if col not in dummies:
        dummies[col] = 0

df = pd.concat([df, dummies], axis=1)

df.to_csv("merged_output_with_stigma-1.csv", index=False)
df.head()
